# Exploring ADORA receptor expression — Q1 walkthrough

**Goal:** map which human cell types express the four adenosine receptors (ADORA1, ADORA2A, ADORA2B, ADORA3) — the receptors caffeine blocks.

**Audience:** first time using the [CellxGene Census](https://chanzuckerberg.github.io/cellxgene-census/) API. This notebook walks through:
1. What the Census is (SOMA / TileDB format)
2. How to peek at what's in it
3. The cell metadata schema (`obs`)
4. The gene metadata schema (`var`)
5. Pulling a targeted query as an `AnnData`
6. Understanding what comes back
7. Per-cell-type aggregation
8. A dotplot
9. Saving intermediates

**Why start here:** the Census gives us 60M+ cells across HCA, Tabula Sapiens, and community datasets in a unified schema. Way better than downloading 20 .h5ad files and harmonizing cell type labels by hand.

## 0. Install dependencies

From the repo root, run in a terminal (not in the notebook):

```bash
uv add cellxgene-census scanpy pyarrow
```

This pins them into `pyproject.toml`. Then restart this kernel.

In [1]:
import cellxgene_census
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print(f'cellxgene_census: {cellxgene_census.__version__}')
print(f'scanpy:           {sc.__version__}')
print(f'pandas:           {pd.__version__}')

cellxgene_census: 1.17.0
scanpy:           1.12.1
pandas:           2.3.3


/tmp/ipykernel_206758/4181733322.py:8: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f'scanpy:           {sc.__version__}')


## 1. What is the Census?

The Census is a **SOMA** object stored in **TileDB** format on S3. You don't download it — you query it. Under the hood:

- It's a hierarchy of arrays (cells × genes) plus dataframes (`obs` cell metadata, `var` gene metadata)
- You filter with SQL-like expressions, and only the matching rows/columns get pulled over the wire
- Each release is versioned — `open_soma()` defaults to the latest stable

When you call `open_soma()` you're opening a lazy handle, not loading anything.

In [2]:
census = cellxgene_census.open_soma()
census

The "stable" release is currently 2025-11-08. Specify 'census_version="2025-11-08"' in future calls to open_soma() to ensure data consistency.


<Collection 's3://cellxgene-census-public-us-west-2/cell-census/2025-11-08/soma/' (open for 'r') (3 items)
    'census_data': 's3://cellxgene-census-public-us-west-2/cell-census/2025-11-08/soma/census_data' (unopened)
    'census_info': 's3://cellxgene-census-public-us-west-2/cell-census/2025-11-08/soma/census_info' (unopened)
    'census_spatial_sequencing': 's3://cellxgene-census-public-us-west-2/cell-census/2025-11-08/soma/census_spatial_sequencing' (unopened)>

## 2. Peek at the summary

The Census has two top-level groups:
- `census_info` — release metadata, dataset list, summary stats
- `census_data` — the actual data, one sub-group per organism (`homo_sapiens`, `mus_musculus`)

Let's see what's in it.

In [3]:
summary = census['census_info']['summary'].read().concat().to_pandas()
summary

,soma_joinid,label,value
0,0,census_schema_version,2.4.0
1,1,census_build_date,2025-11-08
2,2,dataset_schema_version,7.0.0
3,3,total_cell_count,217768036
4,4,unique_cell_count,125463259


In [4]:
datasets = census['census_info']['datasets'].read().concat().to_pandas()
print(f'{len(datasets)} datasets in the Census')
datasets[['collection_name','dataset_title','dataset_total_cell_count']].head(10)

1845 datasets in the Census


,collection_name,dataset_title,dataset_total_cell_count
0,High Resolution Slide-seqV2 Spatial Transcript...,Spatial transcriptomics in mouse: Puck_191112_05,10888
1,High Resolution Slide-seqV2 Spatial Transcript...,Spatial transcriptomics in mouse: Puck_191112_08,10250
2,High Resolution Slide-seqV2 Spatial Transcript...,Spatial transcriptomics in mouse: Puck_191109_20,12906
3,High Resolution Slide-seqV2 Spatial Transcript...,Spatial transcriptomics in mouse: Puck_191112_13,15161
4,HTAN/HTAPP Broad - Spatio-molecular dissection...,HTAPP-330-SMP-1082 scRNA-seq,565
5,"Single-Cell, Single-Nucleus, and Spatial RNA S...",Healthy human liver: B cells,146
6,High Resolution Slide-seqV2 Spatial Transcript...,Spatial transcriptomics in mouse: Puck_191109_14,12351
7,A spatial human thymus cell atlas mapped to a ...,thymus scRNA-seq atlas - myeloid p2 subset,843
8,High Resolution Slide-seqV2 Spatial Transcript...,Spatial transcriptomics in mouse: Puck_191109_18,19156
9,Single-cell analysis of human B cell maturatio...,Human tonsil nonlymphoid cells scRNA,363


## 3. The cell metadata (`obs`)

Every cell has a rich set of metadata columns. The most useful for us:

- `cell_type` — fine-grained cell ontology label (e.g. `CD8-positive, alpha-beta memory T cell`)
- `tissue` — specific tissue label (`cerebral cortex`, `heart left ventricle`)
- `tissue_general` — high-level tissue category (`brain`, `heart`)
- `assay` — sequencing technology (`10x 3' v3`, `Smart-seq2`)
- `dataset_id` — which study the cell came from
- `donor_id`, `sex`, `development_stage`, `disease`

Let's sample a small slice so you can see the structure without pulling millions of rows.

In [5]:
human = census['census_data']['homo_sapiens']

obs_sample = human.obs.read(
    column_names=['cell_type','tissue','tissue_general','assay','dataset_id','donor_id','disease'],
    value_filter="tissue_general == 'heart' and disease == 'normal'",
).concat().to_pandas()

print(f'{len(obs_sample):,} heart cells (normal tissue)')
obs_sample.head()

4,304,522 heart cells (normal tissue)


,cell_type,tissue,tissue_general,assay,dataset_id,donor_id,disease
0,regular atrial cardiac myocyte,atrioventricular node,heart,10x multiome,680029ee-29b3-4e03-bdb3-47d163540ef0,AH1,normal
1,regular atrial cardiac myocyte,atrioventricular node,heart,10x multiome,680029ee-29b3-4e03-bdb3-47d163540ef0,AH1,normal
2,regular atrial cardiac myocyte,atrioventricular node,heart,10x multiome,680029ee-29b3-4e03-bdb3-47d163540ef0,AH1,normal
3,regular atrial cardiac myocyte,atrioventricular node,heart,10x multiome,680029ee-29b3-4e03-bdb3-47d163540ef0,AH1,normal
4,regular atrial cardiac myocyte,atrioventricular node,heart,10x multiome,680029ee-29b3-4e03-bdb3-47d163540ef0,A61,normal


In [6]:
# What cell types are in the heart?
obs_sample['cell_type'].value_counts().head(15)

cell_type
regular ventricular cardiac myocyte    676980
cardiac muscle cell                    603836
fibroblast                             564999
pericyte                               325317
mural cell                             243745
endothelial cell                       242266
capillary endothelial cell             211045
regular atrial cardiac myocyte         179200
myeloid cell                           146172
endocardial cell                       121261
fibroblast of cardiac tissue           119352
smooth muscle cell                      84412
ventricular cardiac muscle cell         83620
endothelial cell of artery              74523
cardiac endothelial cell                72805
Name: count, dtype: int64

In [7]:
# Which datasets contribute to the heart?
obs_sample.groupby('dataset_id').size().sort_values(ascending=False).head()

/tmp/ipykernel_1887772/1189922527.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_sample.groupby('dataset_id').size().sort_values(ascending=False).head()


dataset_id
364bd0c7-f7fd-48ed-99c1-ae26872b1042    931012
d567b692-c374-4628-a508-8008f6778f22    704296
d4e69e01-3ba2-4d6b-a15d-e7048f78f22e    486134
65badd7a-9262-4fd1-9ce2-eb5dc0ca8039    282372
f1606894-59df-4794-a37f-baa7c6fb6de1    217167
dtype: int64

## 4. The gene metadata (`var`)

Genes live under `human.ms['RNA'].var`. Two key columns:

- `feature_id` — Ensembl ID (what the Census actually uses as the key)
- `feature_name` — gene symbol (what humans use)

Always filter on `feature_name` but remember the column IDs in the returned AnnData use Ensembl.

In [8]:
var_df = human.ms['RNA'].var.read().concat().to_pandas()
print(f'{len(var_df):,} genes in the Census')
adora = var_df[var_df['feature_name'].isin(['ADORA1','ADORA2A','ADORA2B','ADORA3'])]
adora

61,497 genes in the Census


,soma_joinid,feature_id,feature_name,feature_type,feature_length,nnz,n_measured_obs
658,658,ENSG00000282608,ADORA3,protein_coding,1067,1069204,135281408
14173,14173,ENSG00000128271,ADORA2A,protein_coding,691,2891447,146634203
15755,15755,ENSG00000163485,ADORA1,protein_coding,2219,10995053,158258689
16407,16407,ENSG00000170425,ADORA2B,protein_coding,1698,3884634,158703591


## 5. Loading the cached AnnData subset

Live `cellxgene_census.get_anndata()` calls can stream a very large number of cell rows from the remote Census, even when we only request four genes. To keep VS Code responsive, this notebook now treats downloads as an offline step.

Run `fetch_adora_cache.py` from a detached shell or overnight job first. Then this notebook loads the local `.h5ad` cache from disk.

Expected cache files:

- `cache/adora_all_tissues.h5ad`: combined cache across completed tissues
- `cache/adora_<tissue>.h5ad`: per-tissue cache, useful for a smaller first pass

If no cache exists, stop here and run the fetch script rather than pulling from the live API inside the notebook.


In [ ]:
from pathlib import Path
import anndata as ad

cache_dirs = [
    Path('cache'),
    Path('lab/001_adora_expression/cache'),
]

cache_path = None
for cache_dir in cache_dirs:
    for candidate in [
        cache_dir / 'adora_all_tissues.h5ad',
        cache_dir / 'adora_brain.h5ad',
    ]:
        if candidate.exists():
            cache_path = candidate
            break
    if cache_path is not None:
        break

if cache_path is None:
    raise FileNotFoundError(
        "No ADORA cache found. Run this from lab/001_adora_expression first:\\n"
        "  python fetch_adora_cache.py --tissues brain\\n"
        "or for the full default run:\\n"
        "  python fetch_adora_cache.py"
    )

adata = ad.read_h5ad(cache_path)
print(f"Loaded {cache_path} with shape {adata.shape}")
adata


## 6. Understanding the AnnData

An `AnnData` is:
- `.X` — cells × genes matrix (sparse CSR by default from Census)
- `.obs` — pandas DataFrame of cell metadata (rows aligned to `.X` rows)
- `.var` — pandas DataFrame of gene metadata (rows aligned to `.X` columns)
- `.obsm`, `.varm`, `.uns` — slots for derived matrices and arbitrary metadata (unused here)

**Values:** Census `.X` is raw integer counts. We'll normalize with scanpy before comparing across cells.

In [ ]:
print(f'Shape: {adata.shape} (cells × genes)')
print(f'X type: {type(adata.X).__name__}')
print(f'X dtype: {adata.X.dtype}')
print(f'X sum (total counts): {adata.X.sum():,.0f}')
print(f'\nFirst 3 cells × 4 genes:')
pd.DataFrame(adata.X[:3].toarray(), columns=adata.var['feature_name'].values)

In [ ]:
print('.obs (cell metadata):')
adata.obs.head()

In [ ]:
print('.var (gene metadata):')
adata.var

## 7. Normalize + map Ensembl → symbol

Two bits of housekeeping before any analysis:
1. Normalize to a constant library size + log1p (so one cell with 10× more reads doesn't dominate)
2. Rename the `.var_names` from Ensembl IDs to gene symbols so plots are readable

In [ ]:
# Normalize
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Rename var_names to gene symbols
adata.var_names = adata.var['feature_name'].values
adata

## 8. Per-cell-type aggregation

For Q1, we want two numbers per (cell type × gene):
- **Mean expression** among all cells of that type
- **% of cells expressing** (count > 0)

The pair gives you a dotplot's inputs.

In [ ]:
genes = ['ADORA1','ADORA2A','ADORA2B','ADORA3']
X = adata[:, genes].X.toarray()
ct = adata.obs['cell_type'].values

df = pd.DataFrame(X, columns=genes)
df['cell_type'] = ct

mean_expr = df.groupby('cell_type').mean()
pct_expr  = df.groupby('cell_type').apply(lambda g: (g[genes] > 0).mean() * 100)

# Keep cell types with >= 50 cells to avoid noise
counts = df['cell_type'].value_counts()
keep = counts[counts >= 50].index
mean_expr = mean_expr.loc[keep]
pct_expr  = pct_expr.loc[keep]

print(f'{len(keep)} cell types with >=50 cells')
mean_expr.head()

In [ ]:
# Top 10 brain cell types for each receptor by mean expression
for g in genes:
    print(f'\n== Top 10 {g} ==')
    top = mean_expr[g].sort_values(ascending=False).head(10)
    for ct, val in top.items():
        pct = pct_expr.loc[ct, g]
        print(f'  {val:6.3f}  ({pct:4.1f}% expressing)  {ct}')

## 9. Dotplot

Scanpy's `sc.pl.dotplot` takes the AnnData directly and groups by any obs column. Size = % expressing, color = mean expression.

In [ ]:
# Restrict to cell types with >= 50 cells for clarity
adata_sub = adata[adata.obs['cell_type'].isin(keep)].copy()

sc.pl.dotplot(
    adata_sub,
    var_names=genes,
    groupby='cell_type',
    standard_scale='var',  # normalize per gene so patterns are visible
    figsize=(5, max(6, len(keep) * 0.25)),
    save='_adora_brain.png',
)

## 10. Save intermediates

Cache the pseudobulk results so you don't have to re-query the Census every time.

In [ ]:
from pathlib import Path
out = Path('.').resolve()

pseudobulk = mean_expr.join(pct_expr, lsuffix='_mean', rsuffix='_pct')
pseudobulk = pseudobulk.reset_index()
pseudobulk['tissue_general'] = 'brain'
pseudobulk.to_feather(out / 'pseudobulk_brain.feather')
print(f'Wrote {out / "pseudobulk_brain.feather"} ({len(pseudobulk)} rows)')

## Where to go next

- **Expand tissues:** repeat sections 5–10 for `heart`, `adipose tissue`, `liver`, `kidney`, `blood`, `intestine`. A loop + concatenation is fine; watch memory.
- **Cross-receptor co-expression:** cells expressing ≥2 receptors may be most caffeine-sensitive. Add a per-cell binary matrix + count.
- **GTEx sanity check:** pseudobulk cell-type means to tissue means, compare against GTEx v8 bulk TPM for the same receptors. Agreement should be directionally consistent.
- **Promote findings:** add interpretation to `entry.md` and a row to the project's eventual summary table.

### Things I glossed over

- **Normalization choice:** I used library-size + log1p. For cross-dataset comparison you may want to use the Census's `normalized` layer directly or run `scvi-tools` for integration.
- **Cell type ontology:** `cell_type` labels are Cell Ontology terms — granularity varies by dataset. Coarser grouping (`cell_type_ontology_term_id` and rolling up) may give cleaner signal.
- **Doublets, QC:** Census has already been QC'd by submitters, but aggressive filtering (`total_counts`, `pct_mito`) isn't applied consistently. For a rigorous run add filters.
- **Assay mixing:** 10x 3' and Smart-seq2 have different drop-out profiles. For a fair comparison, stratify by `assay`.